# Construction & Preprocessing

In [1]:
import pickle
import networkx as nx
import matplotlib.pyplot as plt
import random
from pprint import pprint

random.seed(0)

def draw(G, with_labels=True, title=None, exclude_isolates=False, save_name=None):
    if exclude_isolates:
        G.remove_nodes_from(list(nx.isolates(G)))
    nx.draw(G, pos=nx.spring_layout(G, seed=0), with_labels=with_labels, node_color='skyblue')
    if title:
        plt.title(title)
    plt.show()
    if save_name:
        plt.savefig(save_name)

def extract_largest_cc(G):
    largest_cc = max(nx.connected_components(G), key=len)
    return G.subgraph(largest_cc).copy()

def extract_largest_wcc(G):
    largest_wcc = max(nx.weakly_connected_components(G), key=len)
    return G.subgraph(largest_wcc).copy()

def make_approximate_image(G, name='graph'):
    A = nx.drawing.nx_agraph.to_agraph(G)
    A.layout(prog='sfdp')  # Choose layout: 'sfdp' is good for large graphs
    A.draw(f'{name}.png')

def write_edge_list(G, path):
    with open(path, 'w') as f:
        for u, v in G.edges():
            f.write(f"{u} {v}\n")

def dataset_statistics(G):
    degrees = dict(G.degree()).values()
    return {
        "Number of Nodes": G.number_of_nodes(),
        "Number of Edges": G.number_of_edges(),
        "Density": nx.density(G),
        "Harmonic Diameter": nx.harmonic_diameter(G),
        "Maximum Degree": max(degrees),
        "Average Degree": sum(degrees) / G.number_of_nodes(),
    }

## PrivDPR Counterexample Constructions

### Star (not used -- invalid by edge-case)

To be slightly generous, we overlook this as a counterexample because trivially, removing the hub of a k-star will cause PrivDPR to generate an empty graph, which is obviously distinguishable from the output generated by the neighbouring k-star dataset.

- 0: $k$-star, where $k = 1000$.
- 1: $k$-star with the hub removed (maximum change deletion).
- 2: $K_{2,k}$ ; A $k$-star with an additional node connected ONLY to all leaf nodes (maximum change addition).

All undirected.

### Construction-a: Pure Counterexample

No assumptions other than that the graph has at least one edge.

- 0: $k$-star ($k = 999$), with the following modifications:
    - Edges point from the star leaves to the central node $v$;
    - Additional non-star leaf $w$, and a directed edge $(v, w)$;
    - Directed edge from $w$ to one star-leaf.
- 1: As above, with all of $v$'s edges (both directions) removed.
- u: undirected
- d: directed

In [2]:
name = "Construction-A"

star_size = 1000

edge_list = [(i, 0) for i in range(1, star_size-1)] + [(0, star_size-1), (star_size-1, 1)]
G = nx.DiGraph(edge_list)
print(G)
pprint(dataset_statistics(G))
with open(f"{name}/nx_adj0d.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list0d.txt")
make_approximate_image(G, f"{name}/{name}0")

H = G.to_undirected()
with open(f"{name}/nx_adj0u.pkl", 'wb') as f:
    pickle.dump(H, f)
write_edge_list(H, f"{name}/edge_list0u.txt")

G.remove_edges_from(list(G.in_edges(0)) + list(G.out_edges(0)))
print(G)
pprint(dataset_statistics(G))
with open(f"{name}/nx_adj1d.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list1d.txt")
make_approximate_image(G, f"{name}/{name}1")

H = G.to_undirected()
with open(f"{name}/nx_adj1u.pkl", 'wb') as f:
    pickle.dump(H, f)
write_edge_list(H, f"{name}/edge_list1u.txt")

DiGraph with 1000 nodes and 1000 edges
{'Average Degree': 2.0,
 'Density': 0.001001001001001001,
 'Harmonic Diameter': 545.2064762597854,
 'Maximum Degree': 999,
 'Number of Edges': 1000,
 'Number of Nodes': 1000}
DiGraph with 1000 nodes and 1 edges
{'Average Degree': 0.002,
 'Density': 1.001001001001001e-06,
 'Harmonic Diameter': 999000.0,
 'Maximum Degree': 1,
 'Number of Edges': 1,
 'Number of Nodes': 1000}


### Construction-B: Random Graph Injection on Construction-a

As per Audit 1, except the directed edge from $w$ to one of the star leaves forms the intersection of the $k$-star and a connected ~50-node Erdos-Renyi random graph with $p = 0.025$.
Meant to show that it's not a proportionality thing

- 0: with $v$
- 1: without $v$
- u: undirected
- d: directed

In [9]:
name = "Construction-B"

star_size = 1000
G = nx.erdos_renyi_graph(n=1000, p=0.025, seed=0, directed=True)
G = extract_largest_wcc(G)
#print("ER Size:", G.number_of_nodes())
G = nx.convert_node_labels_to_integers(G)
xn = G.number_of_nodes()
edge_list = [(i, xn) for i in range(xn+1, xn+star_size-2)] + [(v, xn) for v in [*G.edges()][0]]
G.add_edges_from(edge_list)
print(G)
pprint(dataset_statistics(G))

with open(f"{name}/nx_adj0d.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list0d.txt")
#make_approximate_image(G, f"{name}/{name}0")

H = G.to_undirected()
with open(f"{name}/nx_adj0u.pkl", 'wb') as f:
    pickle.dump(H, f)
write_edge_list(H, f"{name}/edge_list0u.txt")

G.remove_edges_from(list(G.in_edges(xn)) + list(G.out_edges(xn)))
print(G)
pprint(dataset_statistics(G))
with open(f"{name}/nx_adj1d.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list1d.txt")
#make_approximate_image(G, f"{name}/{name}1")

H = G.to_undirected()
with open(f"{name}/nx_adj1u.pkl", 'wb') as f:
    pickle.dump(H, f)
write_edge_list(H, f"{name}/edge_list1u.txt")

DiGraph with 1998 nodes and 26099 edges
{'Average Degree': 26.125125125125127,
 'Density': 0.006541092920662275,
 'Harmonic Diameter': 9.34516089057853,
 'Maximum Degree': 999,
 'Number of Edges': 26099,
 'Number of Nodes': 1998}
DiGraph with 1998 nodes and 25100 edges
{'Average Degree': 25.125125125125127,
 'Density': 0.006290717357317257,
 'Harmonic Diameter': 9.374223000896624,
 'Maximum Degree': 73,
 'Number of Edges': 25100,
 'Number of Nodes': 1998}


### Construction-C: Connected Graph Assumption

Topological order:

w -> 999 vertices -> v

Plus, an edge from w to v.

- 0: As above
- 1: All edges upon v removed EXCEPT for w->v
- d: directed

In [4]:
name = "Construction-C"

star_size = 1000

edge_list = [(i, 0) for i in range(1, star_size)] + [(star_size, i) for i in range(0, star_size)]
G = nx.DiGraph(edge_list)
print(G)
pprint(dataset_statistics(G))

with open(f"{name}/nx_adj0d.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list0d.txt")
#make_approximate_image(G, f"{name}/{name}0")

H = G.to_undirected()
with open(f"{name}/nx_adj0u.pkl", 'wb') as f:
    pickle.dump(H, f)
write_edge_list(H, f"{name}/edge_list0u.txt")

G.remove_edges_from(list(G.in_edges(0)) + list(G.out_edges(0)))
G.add_edge(star_size, 0)
#G.remove_nodes_from([xn])
#G = nx.convert_node_labels_to_integers(G)
print(G)
pprint(dataset_statistics(G))
with open(f"{name}/nx_adj1d.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list1d.txt")
#make_approximate_image(G, f"{name}/{name}1")

H = G.to_undirected()
with open(f"{name}/nx_adj1u.pkl", 'wb') as f:
    pickle.dump(H, f)
write_edge_list(H, f"{name}/edge_list1u.txt")

DiGraph with 1001 nodes and 1999 edges
{'Average Degree': 3.994005994005994,
 'Density': 0.001997002997002997,
 'Harmonic Diameter': 500.7503751875938,
 'Maximum Degree': 1000,
 'Number of Edges': 1999,
 'Number of Nodes': 1001}
DiGraph with 1001 nodes and 1000 edges
{'Average Degree': 1.998001998001998,
 'Density': 0.000999000999000999,
 'Harmonic Diameter': 1001.0,
 'Maximum Degree': 1000,
 'Number of Edges': 1000,
 'Number of Nodes': 1001}


### Construction-D: Random Graph Injection on Construction-C

In [5]:
name = "Construction-D"

star_size = 1000
G = nx.erdos_renyi_graph(n=1000, p=0.025, seed=1, directed=True)
G = extract_largest_wcc(G)
#print("ER Size:", G.number_of_nodes())
G = nx.convert_node_labels_to_integers(G)
w, v = [*G.edges()][0]
edge_list = [(i, v) for i in range(1000, 1000+star_size-1)] + [(w, i) for i in range(1000, 1000+star_size-1)]
G.add_edges_from(edge_list)
print(G)
pprint(dataset_statistics(G))
with open(f"{name}/nx_adj0d.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list0d.txt")
#make_approximate_image(G, f"{name}/{name}0")

H = G.to_undirected()
with open(f"{name}/nx_adj0u.pkl", 'wb') as f:
    pickle.dump(H, f)
write_edge_list(H, f"{name}/edge_list0u.txt")

G.remove_edges_from(list(G.in_edges(v)) + list(G.out_edges(v)))
G.add_edge(w, v)
print(G)
pprint(dataset_statistics(G))
with open(f"{name}/nx_adj1d.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list1d.txt")
#make_approximate_image(G, f"{name}/{name}1")

H = G.to_undirected()
with open(f"{name}/nx_adj1u.pkl", 'wb') as f:
    pickle.dump(H, f)
write_edge_list(H, f"{name}/edge_list1u.txt")

DiGraph with 1999 nodes and 27007 edges
{'Average Degree': 27.020510255127565,
 'Density': 0.006761889453235126,
 'Harmonic Diameter': 3.11488228372752,
 'Maximum Degree': 1056,
 'Number of Edges': 27007,
 'Number of Nodes': 1999}
DiGraph with 1999 nodes and 25952 edges
{'Average Degree': 25.96498249124562,
 'Density': 0.006497743366177583,
 'Harmonic Diameter': 5.525593962391409,
 'Maximum Degree': 1052,
 'Number of Edges': 25952,
 'Number of Nodes': 1999}


### Retweet-LOLGOP (Real Dataset 1)

In [6]:
name = "RT-LOLGOP"

with open(f"{name}/rt_lolgop.edges", 'r') as f:
    edges = set()
    for line in f.readlines():
        line = line.strip()
        #if line[0] == "%": continue
        line = line.split(',')
        x = int(line[0])
        y = int(line[1])
        edges.add((x, y))
            
G = nx.DiGraph(edges)
print(G)
pprint(dataset_statistics(G))
G = nx.convert_node_labels_to_integers(G)
with open(f"{name}/nx_adj0.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list0.txt")
#make_approximate_image(G, f"{name}/{name}0")

v = max([(v, d) for v, d in G.degree()], key=lambda x: x[1])[0]
G.remove_edges_from(list(G.in_edges(v)) + list(G.out_edges(v)))
with open(f"{name}/nx_adj1.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list1.txt")
#make_approximate_image(G, f"{name}/{name}1")
print(G)
pprint(dataset_statistics(G))

DiGraph with 9765 nodes and 10324 edges
{'Average Degree': 2.114490527393753,
 'Density': 0.00010827993278337532,
 'Harmonic Diameter': 22.902950573155277,
 'Maximum Degree': 9213,
 'Number of Edges': 10324,
 'Number of Nodes': 9765}
DiGraph with 9765 nodes and 1111 edges
{'Average Degree': 0.227547363031234,
 'Density': 1.1652363940558889e-05,
 'Harmonic Diameter': 54947.82157676372,
 'Maximum Degree': 41,
 'Number of Edges': 1111,
 'Number of Nodes': 9765}


### Anybeat (Real Dataset 2)

In [7]:
name = "Anybeat"

with open(f"{name}/soc-anybeat.edges", 'r') as f:
    edges = set()
    for line in f.readlines():
        line = line.strip()
        #if line[0] == "%": continue
        line = line.split(' ')
        x = int(line[0])
        y = int(line[1])
        edges.add((x, y))
            
G = nx.DiGraph(edges)
print(G)
pprint(dataset_statistics(G))
G = nx.convert_node_labels_to_integers(G)
with open(f"{name}/nx_adj0.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list0.txt")
#make_approximate_image(G, f"{name}/{name}0")

v = max([(v, d) for v, d in G.degree()], key=lambda x: x[1])[0]
G.remove_edges_from(list(G.in_edges(v)) + list(G.out_edges(v)))
with open(f"{name}/nx_adj1.pkl", 'wb') as f:
    pickle.dump(G, f)
write_edge_list(G, f"{name}/edge_list1.txt")
#make_approximate_image(G, f"{name}/{name}1")
print(G)
pprint(dataset_statistics(G))

DiGraph with 12645 nodes and 67053 edges
{'Average Degree': 10.605456702253855,
 'Density': 0.00041938693064907684,
 'Harmonic Diameter': 4.430822156200112,
 'Maximum Degree': 8705,
 'Number of Edges': 67053,
 'Number of Nodes': 12645}
DiGraph with 12645 nodes and 58348 edges
{'Average Degree': 9.2286279161724,
 'Density': 0.00036494099636872826,
 'Harmonic Diameter': 9.329025203213016,
 'Maximum Degree': 2719,
 'Number of Edges': 58348,
 'Number of Nodes': 12645}
